# Protein Expression Distribution Plots

Generates per-patient expression distribution plots for a given protein across all three normalization types (Intensity, iBAQ, LFQ). Each subplot shows individual measurements per patient-position, per-patient mean lines, standard deviation bands, and the global CV of patient means.

In [ ]:
# ============================================================
# CONFIGURATION - Edit these paths and protein name as needed
# ============================================================
import sys
sys.path.insert(0, '..')
from config import DATA_DIR, CSVS_DIR

CSV_DIR = str(CSVS_DIR / "relevant_dataframes_per_norm_type")

# Set the protein name to plot (must match exactly as it appears in the data)
PROTEIN_NAME = " Thioredoxin"

# Output directory for saved figures (set to None to skip saving)
OUTPUT_DIR = str(DATA_DIR / "figures")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from os.path import join

In [ ]:
def load_protein_data(csv_dir):
    """Load the pre-filtered proteomics tables for all three normalization types."""
    data = {}
    for norm_type in ['Intensity', 'iBAQ', 'LFQ']:
        filepath = join(csv_dir, f'relevant_patients_proteomics_table_{norm_type}.csv')
        data[norm_type] = pd.read_csv(filepath)
        print(f"{norm_type}: {data[norm_type].shape[0]} proteins, {data[norm_type].shape[1]} columns")
    return data


def get_protein_row(protein_name, df):
    """Find a protein row by name."""
    mask = df['Protein names'] == protein_name
    if not mask.any():
        # Try partial match
        mask = df['Protein names'].str.contains(protein_name.strip(), na=False, case=False)
    if not mask.any():
        return None
    return df[mask].iloc[0]


def extract_patient_measurements(protein_row, df, norm_type):
    """Extract per-patient measurements from a protein row."""
    if norm_type == 'LFQ':
        lysis_cols = [c for c in df.columns if c.startswith('LFQ intensity') and c.endswith('L')]
    else:
        lysis_cols = [c for c in df.columns if c.startswith(norm_type) and c.endswith('L')]

    patient_measurements = {}
    for col in lysis_cols:
        if norm_type == 'LFQ':
            patient = col.split(' ')[-1].split('_')[0]
        else:
            patient = col.split(' ')[1].split('_')[0]

        val = protein_row[col]
        if patient not in patient_measurements:
            patient_measurements[patient] = []
        patient_measurements[patient].append(val)

    return patient_measurements

In [ ]:
def plot_protein_expression(protein_name, data_dict, output_dir=None):
    """Create a 3-panel expression distribution plot (one per normalization type)."""
    fig, axes = plt.subplots(3, 1, figsize=(20, 15))
    
    # Get the actual protein name from the data for the title
    display_name = protein_name.strip()
    fig.suptitle(f'Expression Levels for {display_name} Across Patients and Positions', fontsize=14)

    for idx, norm_type in enumerate(['Intensity', 'iBAQ', 'LFQ']):
        df = data_dict[norm_type]
        protein_row = get_protein_row(protein_name, df)

        if protein_row is None:
            print(f"Protein '{protein_name}' not found in {norm_type} data")
            continue

        patient_measurements = extract_patient_measurements(protein_row, df, norm_type)

        # Build flat list of (patient, position_index, value) sorted by patient number
        all_points = []
        for patient in sorted(patient_measurements.keys(), key=int):
            for i, val in enumerate(patient_measurements[patient]):
                all_points.append((patient, i, val))

        x_coords = list(range(len(all_points)))
        y_values = [p[2] for p in all_points]
        labels = [f"({p[0]}, pos{p[1]+1})" for p in all_points]

        # Calculate per-patient stats
        patient_means = []
        patient_ranges = []  # (start_idx, end_idx) in the flat list
        patient_stats = {}
        curr_idx = 0

        for patient in sorted(patient_measurements.keys(), key=int):
            measurements = patient_measurements[patient]
            mean_val = np.mean(measurements)
            std_val = np.std(measurements)
            n = len(measurements)
            cv = (std_val / mean_val) * 100 if mean_val != 0 else 0

            patient_means.append(mean_val)
            start = curr_idx
            end = curr_idx + n - 1
            patient_ranges.append((start, end))
            patient_stats[patient] = {'mean': mean_val, 'std': std_val, 'cv': cv, 'n': n}
            curr_idx += n

        # CV of patient means (global stability score)
        valid_means = [m for m in patient_means if m > 0]
        cv_of_means = (np.std(valid_means) / np.mean(valid_means)) * 100 if valid_means else 0

        # Plot
        ax = axes[idx]
        ax.scatter(x_coords, y_values, c='blue', s=50, alpha=0.7)

        # Mean lines and std bands per patient
        for (start, end), patient in zip(patient_ranges, sorted(patient_measurements.keys(), key=int)):
            stats = patient_stats[patient]
            mean_val = stats['mean']
            std_val = stats['std']

            ax.hlines(mean_val, start, end, colors='red', linestyles='--', alpha=0.7)
            ax.fill_between(range(start, end + 1),
                            [mean_val - std_val] * (end - start + 1),
                            [mean_val + std_val] * (end - start + 1),
                            color='red', alpha=0.1)

        # Overall mean line
        overall_mean = np.mean(y_values)
        ax.axhline(y=overall_mean, color='green', linestyle='-', alpha=0.5, label=f'Overall mean')

        # CV annotation
        ax.text(0.02, 0.98, f'CV of patient means: {cv_of_means:.2f}%',
                transform=ax.transAxes, va='top', fontsize=10,
                bbox=dict(facecolor='white', alpha=0.8))

        # Formatting
        ax.set_xticks(x_coords)
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
        ax.set_title(f'{norm_type} Normalization')
        ax.set_ylabel('Expression Level')
        ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
        ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))

    plt.tight_layout()

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        safe_name = display_name.replace(' ', '_').replace(';', '').replace('/', '_')
        filepath = join(output_dir, f'expression_distribution_{safe_name}.png')
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f'Saved to: {filepath}')

    plt.show()
    plt.close()
    return fig

In [ ]:
# Load data
data_dict = load_protein_data(CSV_DIR)

In [ ]:
# Generate plot for the configured protein
fig = plot_protein_expression(PROTEIN_NAME, data_dict, output_dir=OUTPUT_DIR)

In [ ]:
# ============================================================
# BATCH MODE: Generate plots for all top-20 stable proteins
# ============================================================

TOP20_DIR = str(CSVS_DIR / 'top_20_proteins_per_norm_type')

# Get unique protein names across all norm types
all_proteins = set()
for norm_type in ['Intensity', 'iBAQ', 'LFQ']:
    top20 = pd.read_csv(join(TOP20_DIR, f'top_20_proteins_{norm_type}.csv'))
    all_proteins.update(top20['Protein names'].values)

print(f'Generating plots for {len(all_proteins)} unique proteins...')
for prot in sorted(all_proteins):
    print(f'\n--- {prot.strip()} ---')
    plot_protein_expression(prot, data_dict, output_dir=OUTPUT_DIR)